In [1]:
!pip install faker

In [2]:
!python generate_hospital_data.py

python: can't open file 'C:\\Users\\muluw\\generate_hospital_data.py': [Errno 2] No such file or directory


Fixed 800 and go

In [5]:
from faker import Faker
import random
from datetime import datetime, timedelta

fake = Faker()
random.seed(42)
fake.seed_instance(42)

# ================== CONFIG ==================
NUM_PATIENTS = 300
NUM_DOCTORS = 30
NUM_DEPARTMENTS = 8
NUM_APPOINTMENTS = 800
NUM_MEDICAL_RECORDS = 700
NUM_PRESCRIPTIONS = 1400
NUM_MEDICINES = 80
NUM_BILLINGS = 700
NUM_STAFF = 70
NUM_ROOMS = 35
NUM_ROOM_ASSIGNMENTS = 120

BATCH_SIZE = 800   # Safe limit under 1000
# ===========================================

print("-- HospitalDB Improved Synthetic Data Generator for SQL Server")
print("USE HospitalDB;")
print("SET NOCOUNT ON;")
print("GO")

# ====================== CLEAR EXISTING DATA ======================
print("\n-- Clear existing data to avoid duplicate key errors")
print("DELETE FROM RoomAssignment;")
print("DELETE FROM Prescription;")
print("DELETE FROM MedicalRecords;")
print("DELETE FROM Appointment;")
print("DELETE FROM Billing;")
print("DELETE FROM Room;")
print("DELETE FROM Staff;")
print("DELETE FROM Doctor;")
print("DELETE FROM Patient;")
print("DELETE FROM Department;")
print("DELETE FROM Medicine;")
print("GO")

# Reset Identity columns
print("\n-- Reset Identity counters")
tables = ['Department', 'Patient', 'Doctor', 'Staff', 'Room', 'Medicine', 
          'Appointment', 'MedicalRecords', 'Prescription', 'Billing', 'RoomAssignment']
for table in tables:
    print(f"DBCC CHECKIDENT ('{table}', RESEED, 0);")
print("GO")

# ====================== 1. Department ======================
depts_list = ["Cardiology", "Neurology", "Pediatrics", "Orthopedics", "Emergency", 
              "Oncology", "Gynecology", "Urology"]

print("\n-- 1. Department")
print("INSERT INTO Department (DepartmentName, Location, PhoneExtension) VALUES")
for i in range(NUM_DEPARTMENTS):
    name = depts_list[i % len(depts_list)]
    loc = fake.city() + " Wing"
    ext = f"EXT{random.randint(100,999)}"
    comma = ',' if i < NUM_DEPARTMENTS-1 else ';'
    print(f"    ('{name}', '{loc}', '{ext}'){comma}")
print("GO")

# ====================== Helper Function for Batching ======================
def insert_with_batch(table_name, columns, values_list, batch_size=800):
    if not values_list:
        return
    print(f"\n-- {table_name}")
    print(f"INSERT INTO {table_name} ({columns}) VALUES")
    
    for i, row in enumerate(values_list):
        comma = ',' if i < len(values_list)-1 else ';'
        print(f"    {row}{comma}")
        
        # Add GO after every batch
        if (i + 1) % batch_size == 0 and i < len(values_list)-1:
            print("GO")
            print(f"INSERT INTO {table_name} ({columns}) VALUES")
    
    print("GO")

# ====================== 2. Patient ======================
patient_values = []
for i in range(NUM_PATIENTS):
    fn = fake.first_name().replace("'", "''")
    ln = fake.last_name().replace("'", "''")
    dob = fake.date_of_birth(minimum_age=1, maximum_age=90)
    gender = random.choice(['Male', 'Female'])
    addr = fake.address().replace('\n', ', ').replace("'", "''")
    phone = fake.phone_number()[:20].replace("'", "''")
    email = fake.email().replace("'", "''")
    e_name = fake.name().replace("'", "''")
    e_phone = fake.phone_number()[:20].replace("'", "''")
    patient_values.append(f"('{fn}', '{ln}', '{dob}', '{gender}', '{addr}', '{phone}', '{email}', '{e_name}', '{e_phone}')")

insert_with_batch("Patient", "FirstName, LastName, DateOfBirth, Gender, Address, PhoneNumber, Email, EmergencyContactName, EmergencyContactPhone", patient_values)

# ====================== 3. Doctor ======================
doctor_values = []
for i in range(NUM_DOCTORS):
    fn = fake.first_name().replace("'", "''")
    ln = fake.last_name().replace("'", "''")
    spec = random.choice(["Cardiologist", "Neurologist", "Pediatrician", "Orthopedic Surgeon", 
                          "Emergency Physician", "Oncologist", "Gynecologist", "Urologist"])
    phone = fake.phone_number()[:20].replace("'", "''")
    email = fake.email().replace("'", "''")
    dept_id = random.randint(1, NUM_DEPARTMENTS)
    avail = random.choice(['Available', 'Busy', 'OnLeave'])
    doctor_values.append(f"('{fn}', '{ln}', '{spec}', '{phone}', '{email}', {dept_id}, '{avail}')")

insert_with_batch("Doctor", "FirstName, LastName, Specialization, PhoneNumber, Email, DepartmentID, Availability", doctor_values)

# ====================== 4. Staff ======================
staff_values = []
for i in range(NUM_STAFF):
    fn = fake.first_name().replace("'", "''")
    ln = fake.last_name().replace("'", "''")
    role = random.choice(["Nurse", "Registered Nurse", "Admin Assistant", "Receptionist", "Lab Technician", "Pharmacist"])
    dept_id = random.randint(1, NUM_DEPARTMENTS) if random.random() > 0.15 else 'NULL'
    phone = fake.phone_number()[:20].replace("'", "''")
    email = fake.email().replace("'", "''")
    shift = random.choice(['Morning (7-3)', 'Evening (3-11)', 'Night (11-7)'])
    staff_values.append(f"('{fn}', '{ln}', '{role}', {dept_id}, '{phone}', '{email}', '{shift}')")

insert_with_batch("Staff", "FirstName, LastName, Role, DepartmentID, PhoneNumber, Email, ShiftHours", staff_values)

# ====================== 5. Room ======================
room_values = []
for i in range(NUM_ROOMS):
    room_num = f"R{100 + i}"
    dept_id = random.randint(1, NUM_DEPARTMENTS)
    rtype = random.choice(['General', 'Private', 'ICU', 'Semi-Private'])
    status = random.choice(['Available', 'Occupied', 'Maintenance'])
    room_values.append(f"('{room_num}', {dept_id}, '{rtype}', '{status}')")

insert_with_batch("Room", "RoomNumber, DepartmentID, RoomType, AvailabilityStatus", room_values)

# ====================== 6. Medicine ======================
medicine_values = []
for i in range(NUM_MEDICINES):
    name = random.choice(["Paracetamol", "Amoxicillin", "Ibuprofen", "Omeprazole", "Metformin", "Amlodipine"]) + f" {fake.word().capitalize()}"
    manuf = fake.company().replace("'", "''")
    stock = random.randint(50, 800)
    price = round(random.uniform(4.99, 189.99), 2)
    medicine_values.append(f"('{name}', '{manuf}', {stock}, {price})")

insert_with_batch("Medicine", "MedicineName, Manufacturer, StockQuantity, Price", medicine_values)

# ====================== 7. Appointment ======================
appointment_values = []
start_date = datetime.now() - timedelta(days=180)
for i in range(NUM_APPOINTMENTS):
    p_id = random.randint(1, NUM_PATIENTS)
    d_id = random.randint(1, NUM_DOCTORS)
    dept_id = random.randint(1, NUM_DEPARTMENTS)
    app_date = start_date + timedelta(days=random.randint(0, 180))
    app_time = f"{random.randint(8,16):02d}:{random.choice(['00','15','30','45'])}"
    status = random.choices(['Scheduled', 'Completed', 'Cancelled'], weights=[0.7, 0.25, 0.05])[0]
    appointment_values.append(f"({p_id}, {d_id}, {dept_id}, '{app_date.date()}', '{app_time}', '{status}')")

insert_with_batch("Appointment", "PatientID, DoctorID, DepartmentID, AppointmentDate, AppointmentTime, Status", appointment_values, batch_size=800)

# ====================== 8. MedicalRecords ======================
record_values = []
for i in range(NUM_MEDICAL_RECORDS):
    p_id = random.randint(1, NUM_PATIENTS)
    d_id = random.randint(1, NUM_DOCTORS)
    visit_date = datetime.now() - timedelta(days=random.randint(0, 365))
    diagnosis = random.choice(["Hypertension", "Type 2 Diabetes", "Migraine", "Lower Back Pain", "Pneumonia", "Arthritis"])
    treatment = fake.sentence(nb_words=8).replace("'", "''")
    pres = fake.sentence(nb_words=5).replace("'", "''")
    record_values.append(f"({p_id}, {d_id}, '{visit_date.date()}', '{diagnosis}', '{treatment}', '{pres}')")

insert_with_batch("MedicalRecords", "PatientID, DoctorID, VisitDate, Diagnosis, TreatmentPlan, Prescription", record_values)

# ====================== 9. Prescription ======================
pres_values = []
dosages = ["500mg", "250mg", "1 tablet", "10ml", "5mg"]
frequencies = ["Once daily", "Twice daily", "Three times daily"]
durations = ["7 days", "14 days", "30 days", "5 days"]
for i in range(NUM_PRESCRIPTIONS):
    rec_id = random.randint(1, NUM_MEDICAL_RECORDS)
    med_id = random.randint(1, NUM_MEDICINES)
    dosage = random.choice(dosages)
    freq = random.choice(frequencies)
    dur = random.choice(durations)
    pres_values.append(f"({rec_id}, {med_id}, '{dosage}', '{freq}', '{dur}')")

insert_with_batch("Prescription", "RecordID, MedicineID, Dosage, Frequency, Duration", pres_values, batch_size=800)

# ====================== 10. Billing ======================
billing_values = []
for i in range(NUM_BILLINGS):
    p_id = random.randint(1, NUM_PATIENTS)
    amount = round(random.uniform(50, 2500), 2)
    status = random.choices(['Paid', 'Unpaid'], weights=[0.75, 0.25])[0]
    pay_date = None if status == 'Unpaid' else (datetime.now() - timedelta(days=random.randint(0,90))).date()
    method = random.choice(["Cash", "Credit Card", "Insurance"]) if status == 'Paid' else None
    
    pay_date_str = f"'{pay_date}'" if pay_date else 'NULL'
    method_str = f"'{method}'" if method else 'NULL'
    billing_values.append(f"({p_id}, {amount}, '{status}', {pay_date_str}, {method_str})")

insert_with_batch("Billing", "PatientID, TotalAmount, PaymentStatus, PaymentDate, PaymentMethod", billing_values)

# ====================== 11. RoomAssignment ======================
room_assign_values = []
for i in range(NUM_ROOM_ASSIGNMENTS):
    room_id = random.randint(1, NUM_ROOMS)
    patient_id = random.randint(1, NUM_PATIENTS)
    admit = datetime.now() - timedelta(days=random.randint(5, 90))
    discharge = admit + timedelta(days=random.randint(1, 15)) if random.random() > 0.3 else None
    discharge_str = f"'{discharge.date()}'" if discharge else 'NULL'
    room_assign_values.append(f"({room_id}, {patient_id}, '{admit.date()}', {discharge_str})")

insert_with_batch("RoomAssignment", "RoomID, PatientID, AdmissionDate, DischargeDate", room_assign_values)

print("\n-- Data generation completed successfully!")
print("-- You can now run your business questions and queries.")

-- HospitalDB Improved Synthetic Data Generator for SQL Server
USE HospitalDB;
SET NOCOUNT ON;
GO

-- Clear existing data to avoid duplicate key errors
DELETE FROM RoomAssignment;
DELETE FROM Prescription;
DELETE FROM MedicalRecords;
DELETE FROM Appointment;
DELETE FROM Billing;
DELETE FROM Room;
DELETE FROM Staff;
DELETE FROM Doctor;
DELETE FROM Patient;
DELETE FROM Department;
DELETE FROM Medicine;
GO

-- Reset Identity counters
DBCC CHECKIDENT ('Department', RESEED, 0);
DBCC CHECKIDENT ('Patient', RESEED, 0);
DBCC CHECKIDENT ('Doctor', RESEED, 0);
DBCC CHECKIDENT ('Staff', RESEED, 0);
DBCC CHECKIDENT ('Room', RESEED, 0);
DBCC CHECKIDENT ('Medicine', RESEED, 0);
DBCC CHECKIDENT ('Appointment', RESEED, 0);
DBCC CHECKIDENT ('MedicalRecords', RESEED, 0);
DBCC CHECKIDENT ('Prescription', RESEED, 0);
DBCC CHECKIDENT ('Billing', RESEED, 0);
DBCC CHECKIDENT ('RoomAssignment', RESEED, 0);
GO

-- 1. Department
INSERT INTO Department (DepartmentName, Location, PhoneExtension) VALUES
    ('Card